# Triton Softmax 课后练习

本练习要求你实现支持超大 `n_cols` 的 Softmax 算子，使用两阶段 Reduction 技术。

## 背景

在教程中我们实现的 Softmax 要求 `BLOCK_SIZE >= n_cols`，这意味着列数不能太大（受限于 GPU 寄存器数量）。

当 `n_cols` 超过单个 block 能处理的范围时，需要将每行分成多个 block：

1. **Stage 1**：每个 block 处理行的一段，计算局部的 (max, sum)
2. **Stage 2**：合并所有 block 的结果，计算最终 softmax

## 任务

实现两阶段 Reduction 的 Softmax，支持任意大小的 `n_cols`。

In [ ]:
import torch
import triton
import triton.language as tl
import time  # 添加 time 模块用于性能测试

@triton.jit
def _softmax_stage1_kernel(
    x_ptr,
    partial_max_ptr,
    partial_sum_ptr,
    n_rows, n_cols,
    BLOCK_SIZE: tl.constexpr,
):
    """
    Stage 1: 每个 block 处理行的一段，计算局部的 max 和 sum
    
    TODO: 实现这个 kernel
    
    提示：
    - 使用 tl.program_id(0) 获取全局 block ID
    - 计算该 block 负责的行号和列范围
    - 加载数据并计算该段的局部 max 和 sum
    - 将结果写入 partial_max_ptr 和 partial_sum_ptr
    """
    pass

@triton.jit
def _softmax_stage2_kernel(
    x_ptr,
    partial_max_ptr,
    partial_sum_ptr,
    output_ptr,
    n_rows, n_cols, n_blocks_per_row,
    BLOCK_SIZE: tl.constexpr,
):
    """
    Stage 2: 合并所有 block 的结果
    
    TODO: 实现这个 kernel
    
    提示：
    - 每个 program 处理一行
    - 加载该行所有 block 的局部 (max, sum)
    - 计算 global max：max of all partial_max
    - 计算全局指数和：sum of exp(x - global_max)
    - 这需要重新加载原始数据（或保存中间结果）
    """
    pass

def softmax_large(x: torch.Tensor) -> torch.Tensor:
    """
    支持超大 n_cols 的 Softmax（两阶段 Reduction）
    
    Args:
        x: 输入张量，shape [M, N]，N 可以任意大
    
    Returns:
        输出张量，shape [M, N]
    """
    pass

## 测试

完成实现后，运行以下测试验证正确性：

In [ ]:

def benchmark_softmax():
    """
    性能对比 benchmark：两阶段 softmax vs 普通 softmax vs PyTorch
    """
    print("=" * 60)
    print("Softmax Performance Benchmark")
    print("=" * 60)

    # 测试不同的矩阵大小
    # 可以适当根据自己的显卡规模来减小规模
    test_configs = [
        (1024, 512),
        (1024, 1024),
        (1024, 2048),
        (1024, 4096),
        (1024, 8192),
        (1024, 16384),
        (1024, 32768),
        (1024, 65536),
    ]

    warmup_runs = 3
    benchmark_runs = 10

    for n_rows, n_cols in test_configs:
        print(f"\nMatrix size: {n_rows} x {n_cols}")
        print("-" * 40)

        # 生成测试数据
        x = torch.randn(n_rows, n_cols, device='cuda', dtype=torch.float32)

        # PyTorch 基准测试
        torch.cuda.synchronize()
        # Warmup
        for _ in range(warmup_runs):
            _ = torch.nn.functional.softmax(x, dim=-1)
        torch.cuda.synchronize()

        # Benchmark
        start_time = time.time()
        for _ in range(benchmark_runs):
            y_torch = torch.nn.functional.softmax(x, dim=-1)
        torch.cuda.synchronize()
        torch_time = (time.time() - start_time) / benchmark_runs * 1000  # ms

        print(f"PyTorch:          {torch_time:.3f} ms")

        # 普通 Triton softmax（如果可以处理的话）
        try:
            torch.cuda.synchronize()
            # Warmup
            for _ in range(warmup_runs):
                _ = softmax_naive(x)
            torch.cuda.synchronize()

            # Benchmark
            start_time = time.time()
            for _ in range(benchmark_runs):
                y_naive = softmax_naive(x)
            torch.cuda.synchronize()
            naive_time = (time.time() - start_time) / benchmark_runs * 1000  # ms

            # 验证正确性
            max_error = torch.max(torch.abs(y_naive - y_torch)).item()
            print(f"Naive Triton:     {naive_time:.3f} ms (error: {max_error:.2e})")

        except Exception as e:
            print(f"Naive Triton:     Failed ({str(e)})")


        # 两阶段 Triton softmax
        torch.cuda.synchronize()
        # Warmup
        for _ in range(warmup_runs):
            _ = softmax_large(x)
        torch.cuda.synchronize()

        # Benchmark
        start_time = time.time()
        for _ in range(benchmark_runs):
            y_large = softmax_large(x)
        torch.cuda.synchronize()
        large_time = (time.time() - start_time) / benchmark_runs * 1000  # ms

        # 验证正确性
        max_error = torch.max(torch.abs(y_large - y_torch)).item()
        print(f"Two-stage Triton: {large_time:.3f} ms (error: {max_error:.2e})")

        # 计算加速比
        speedup_vs_torch = torch_time / large_time
        print(f"Speedup vs PyTorch: {speedup_vs_torch:.2f}x")

        if n_cols <= 2048:
            try:
                speedup_vs_naive = naive_time / large_time
                print(f"Speedup vs Naive: {speedup_vs_naive:.2f}x")
            except:
                pass


print("\n" + "=" * 60)
print("Running Performance Benchmark...")
benchmark_softmax()

## 提示

### Stage 1 Kernel

1. 计算全局 block ID，分解为行号和 block 号
   ```python
   block_id = tl.program_id(0)
   row_idx = block_id // n_blocks_per_row
   block_idx = block_id % n_blocks_per_row
   ```

2. 计算该 block 负责的列范围
   ```python
   col_start = block_idx * BLOCK_SIZE
   col_offsets = tl.arange(0, BLOCK_SIZE)
   ```

3. 加载数据并计算局部 max 和 sum

### Stage 2 Kernel

1. 先计算 global max（所有 partial_max 的最大值）

2. 重新加载原始数据（或保存中间 exp 结果），计算全局指数和

3. 归一化并写回结果

### 性能考虑

- Stage 2 需要重新加载原始数据，这是额外的全局内存访问
- 可以考虑在 Stage 1 保存 `exp(x - local_max)` 的结果，但这会增加显存占用
- 这是一个典型的 **space-time tradeoff**

## 作业答案

完成练习后，可以参考以下完整实现：

### Stage 1 Kernel 实现

```python
@triton.jit
def _softmax_stage1_kernel(
    x_ptr,
    partial_max_ptr,
    partial_sum_ptr,
    n_rows, n_cols,
    BLOCK_SIZE: tl.constexpr,
):
    # 获取全局 block ID
    block_id = tl.program_id(0)

    # 计算每行需要的 block 数量
    n_blocks_per_row = tl.cdiv(n_cols, BLOCK_SIZE)

    # 计算当前 block 负责的行号和列 block 编号
    row_id = block_id // n_blocks_per_row
    col_block_id = block_id % n_blocks_per_row

    # 计算列偏移范围
    col_start = col_block_id * BLOCK_SIZE
    col_offsets = col_start + tl.arange(0, BLOCK_SIZE)

    # 创建 mask 确保不超出边界
    mask = col_offsets < n_cols

    # 计算输入数据的内存地址
    row_start_ptr = x_ptr + row_id * n_cols
    x_ptrs = row_start_ptr + col_offsets

    # 加载数据，对于超出边界的位置使用 -inf
    x = tl.load(x_ptrs, mask=mask, other=-float('inf'))

    # 计算局部最大值
    local_max = tl.max(x, axis=0)

    # 计算 exp(x - local_max) 并求和
    x_shifted = x - local_max
    exp_x = tl.exp(x_shifted)
    local_sum = tl.sum(exp_x, axis=0)

    # 计算输出地址并存储结果
    output_idx = row_id * n_blocks_per_row + col_block_id
    tl.store(partial_max_ptr + output_idx, local_max)
    tl.store(partial_sum_ptr + output_idx, local_sum)
```

### Stage 2 Kernel 实现

```python
@triton.jit
def _softmax_stage2_kernel(
    x_ptr,
    partial_max_ptr,
    partial_sum_ptr,
    output_ptr,
    n_rows, n_cols, n_blocks_per_row,
    BLOCK_SIZE: tl.constexpr,
):
    # 每个 program 处理一行
    row_id = tl.program_id(0)

    # 使用循环逐个加载局部结果并计算全局 max 和 sum
    global_max = -float('inf')
    global_sum = 0.0

    # 第一遍：找到全局最大值
    for block_idx in range(n_blocks_per_row):
        partial_idx = row_id * n_blocks_per_row + block_idx
        local_max = tl.load(partial_max_ptr + partial_idx)
        global_max = tl.maximum(global_max, local_max)

    # 第二遍：计算调整后的指数和
    for block_idx in range(n_blocks_per_row):
        partial_idx = row_id * n_blocks_per_row + block_idx
        local_max = tl.load(partial_max_ptr + partial_idx)
        local_sum = tl.load(partial_sum_ptr + partial_idx)

        # 调整局部和：adjusted_sum = local_sum * exp(local_max - global_max)
        max_diff = local_max - global_max
        exp_max_diff = tl.exp(max_diff)
        adjusted_sum = local_sum * exp_max_diff
        global_sum += adjusted_sum

    # 重新加载原始数据并计算最终的 softmax 结果
    row_start_ptr = x_ptr + row_id * n_cols
    output_row_start_ptr = output_ptr + row_id * n_cols

    # 处理每个 block 的数据
    for block_idx in range(n_blocks_per_row):
        # 计算列偏移范围
        col_start = block_idx * BLOCK_SIZE
        col_offsets = col_start + tl.arange(0, BLOCK_SIZE)

        # 创建 mask 确保不超出边界
        mask = col_offsets < n_cols

        # 加载原始数据
        x_ptrs = row_start_ptr + col_offsets
        x = tl.load(x_ptrs, mask=mask, other=0.0)

        # 计算最终的 softmax 结果
        x_shifted = x - global_max
        exp_x = tl.exp(x_shifted)
        softmax_result = exp_x / global_sum

        # 存储结果
        output_ptrs = output_row_start_ptr + col_offsets
        tl.store(output_ptrs, softmax_result, mask=mask)
```

### Host 端函数实现

```python
def softmax_large(x: torch.Tensor) -> torch.Tensor:
    n_rows, n_cols = x.shape
    output = torch.empty_like(x)

    BLOCK_SIZE = 512  # 固定 block 大小
    n_blocks_per_row = triton.cdiv(n_cols, BLOCK_SIZE)
    total_blocks = n_rows * n_blocks_per_row

    # 分配中间结果存储
    partial_max = torch.empty(n_rows, n_blocks_per_row, device='cuda', dtype=torch.float32)
    partial_sum = torch.empty(n_rows, n_blocks_per_row, device='cuda', dtype=torch.float32)

    # Stage 1: 计算局部结果
    grid = (total_blocks,)
    _softmax_stage1_kernel[grid](
        x, partial_max, partial_sum,
        n_rows, n_cols,
        BLOCK_SIZE=BLOCK_SIZE,
    )

    # Stage 2: 合并结果
    grid = (n_rows,)
    _softmax_stage2_kernel[grid](
        x, partial_max, partial_sum, output,
        n_rows, n_cols, n_blocks_per_row,
        BLOCK_SIZE=BLOCK_SIZE,
    )

    return output
```